In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz
from pathlib import Path

# filenames (located in current working directory)
metric_file = Path("metric.csv")
trace_file = Path("trace.csv")
log_file = Path("log.csv")
error_file = Path("error_logs.csv")

# Load dataframes (keep them in memory for later steps)
df_metric = pd.read_csv(metric_file) if metric_file.exists() else pd.DataFrame()
df_trace = pd.read_csv(trace_file) if trace_file.exists() else pd.DataFrame()
df_log = pd.read_csv(log_file) if log_file.exists() else pd.DataFrame()
df_error = pd.read_csv(error_file) if error_file.exists() else pd.DataFrame()

# Standardize KPI column name for numeric telemetry files
if not df_metric.empty and 'kpi_name' in df_metric.columns:
    df_metric = df_metric.rename(columns={'kpi_name': 'KPI_name'})
if not df_trace.empty and 'trace_name' in df_trace.columns:
    df_trace = df_trace.rename(columns={'trace_name': 'KPI_name'})
if not df_log.empty and 'log_name' in df_log.columns:
    df_log = df_log.rename(columns={'log_name': 'KPI_name'})

# Function to aggregate per (cmdb_id, KPI_name)
def aggregate_numeric(df):
    if df.empty:
        return pd.DataFrame(columns=['cmdb_id','KPI_name','count','min','p50','p95','max','mean'])
    # ensure value is numeric
    df = df.copy()
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    agg = df.groupby(['cmdb_id','KPI_name'], as_index=False)['value'].agg(
        count = 'size',
        min = 'min',
        p50 = lambda x: x.quantile(0.5),
        p95 = lambda x: x.quantile(0.95),
        max = 'max',
        mean = 'mean'
    )
    # round numeric columns for compact display
    for c in ['min','p50','p95','max','mean']:
        if c in agg:
            agg[c] = agg[c].round(6)
    # sort by count desc
    agg = agg.sort_values('count', ascending=False).reset_index(drop=True)
    return agg

# Aggregate each numeric file
metric_agg = aggregate_numeric(df_metric)
trace_agg = aggregate_numeric(df_trace)
log_agg = aggregate_numeric(df_log)

# Top 20 rows for each
metric_agg_top20 = metric_agg.head(20)
trace_agg_top20 = trace_agg.head(20)
log_agg_top20 = log_agg.head(20)

# Distinct cmdb_id values in each numeric file (compact)
metric_cmdbs = pd.Series(sorted(df_metric['cmdb_id'].dropna().unique())) if not df_metric.empty else pd.Series(dtype=object)
trace_cmdbs = pd.Series(sorted(df_trace['cmdb_id'].dropna().unique())) if not df_trace.empty else pd.Series(dtype=object)
log_cmdbs = pd.Series(sorted(df_log['cmdb_id'].dropna().unique())) if not df_log.empty else pd.Series(dtype=object)

# Process error_logs: count per cmdb_id and earliest timestamp (use UTC+8)
error_summary = pd.DataFrame(columns=['cmdb_id','count','earliest_timestamp_Asia/Shanghai'])
if not df_error.empty:
    df_error = df_error.copy()
    # convert epoch seconds to timezone-aware datetime in Asia/Shanghai (UTC+8)
    tz = pytz.timezone('Asia/Shanghai')
    df_error['ts_utc'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)
    df_error['ts_shanghai'] = df_error['ts_utc'].dt.tz_convert(tz)
    grp = df_error.groupby('cmdb_id', as_index=False).agg(
        count = ('timestamp','size'),
        earliest_ts = ('ts_shanghai','min')
    )
    grp = grp.rename(columns={'earliest_ts':'earliest_timestamp_Asia/Shanghai'})
    # sort by count desc and limit to top 20
    error_summary = grp.sort_values('count', ascending=False).reset_index(drop=True).head(20)

# Display compact results (variables for the controller)
metric_agg_top20, metric_cmdbs, trace_agg_top20, trace_cmdbs, log_agg_top20, log_cmdbs, error_summary

```
Out[1]:
```
```python
# Build a plain-English summary string based on previously computed variables in the kernel:
# metric_agg_top20, trace_agg_top20, log_agg_top20, metric_cmdbs, trace_cmdbs, log_cmdbs, error_summary

def top_lines_summary(df, n=3):
    if df.empty:
        return "  (no data)\n"
    lines = []
    for i, row in df.head(n).iterrows():
        k = row.get('KPI_name', '') or row.get('kpi_name', '')
        lines.append(f"  - {row['cmdb_id']} | {k[:120]} | count={int(row['count'])}, mean={row['mean']}")
    return "\n".join(lines) + "\n"

metric_cmdb_list = list(metric_cmdbs.head(20)) if 'metric_cmdbs' in globals() else []
trace_cmdb_list = list(trace_cmdbs.head(20)) if 'trace_cmdbs' in globals() else []
log_cmdb_list = list(log_cmdbs.head(20)) if 'log_cmdbs' in globals() else []

summary_lines = [
    "Telemetry summary (compact):",
    "",
    "1) Metrics (metric.csv) — Top groups by point count:",
    top_lines_summary(metric_agg_top20, n=5),
    f"Distinct cmdb_id count in metric.csv: {len(metric_cmdbs)}. Sample up to 20: {metric_cmdb_list}",
    "",
    "2) Traces (trace.csv) — Top groups by point count:",
    top_lines_summary(trace_agg_top20, n=5),
    f"Distinct cmdb_id count in trace.csv: {len(trace_cmdbs)}. Sample up to 20: {trace_cmdb_list}",
    "",
    "3) Logs (log.csv) — Top groups by point count:",
    top_lines_summary(log_agg_top20, n=6),
    f"Distinct cmdb_id count in log.csv: {len(log_cmdbs)}. Sample up to 20: {log_cmdb_list}",
    "",
    "4) Error logs (error_logs.csv):",
    ("  - No rows found." if (('error_summary' in globals()) and error_summary.empty) else
     ("  - Top cmdb_id error counts (up to 20):\n" +
      "\n".join([f\"    * {r['cmdb_id']} | count={int(r['count'])} | earliest={r['earliest_timestamp_Asia/Shanghai']}\" for _, r in error_summary.iterrows()]))),
    "",
    "Notes:",
    " - All summaries show: cmdb_id, KPI name (truncated), count, and mean where available.",
    " - error_logs.csv returned no entries in this run.",
]

summary_string = "\n".join(summary_lines)

# Display the summary string (compact, plain English)
summary_string
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           KPI_name  count           min           p50           p95           max          mean
0       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  6.700000e+01  7.400000e+01  7.600000e+01  7.600000e+01  7.332258e+01
1   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  0.000000e+00  0.000000e+00  1.000000e+00  2.000000e+00  1.935480e-01
2   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  5.000000e+02  5.000000e+02  5.000000e+02  5.000000e+02  5.000000e+02
3   Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
4       MG02                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  7.627892e+06  7.628796e+06  7.629611e+06  7.629701e+06  7.628796e+06
5   Tomcat03                        OSLinux-CPU_CPU_CPUUserTime     31  2.519220e+01  2.531488e+01  9.859705e+01  9.911900e+01  3.700192e+01
6   Tomcat03                             OSLinux-CPU_CPU_CPUWio     31  0.000000e+00  4.989600e-02  1.411030e+00  5.004400e+00  3.199110e-01
7   Tomcat03                        OSLinux-CPU_CPU_CPUidleutil     31  3.722500e-02  7.396290e+01  7.419153e+01  7.421650e+01  6.201316e+01
8   Tomcat03                            OSLinux-CPU_CPU_CPULoad     31  1.000000e+00  1.090000e+00  9.165000e+00  9.690000e+00  2.420000e+00
9   Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
10  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
11  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
12  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02
13      IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  5.811566e+07  5.811566e+07  5.811566e+07  5.811566e+07  5.811566e+07
14      IG02           JVM-Threads_7778_JVM_ThreadCount_Threads     31  6.800000e+01  7.300000e+01  7.600000e+01  7.600000e+01  7.296774e+01
15  Tomcat03  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31  3.000000e-01  3.000000e-01  3.000000e-01  4.000000e-01  3.032260e-01
16  Tomcat03         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
17  Tomcat03      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00
18  Tomcat03     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  0.000000e+00  0.000000e+00  5.000000e-01  1.000000e+00  6.451600e-02
19  Tomcat03                         OSLinux-CPU_CPU_CPUSysTime     31  2.993000e-01  4.012080e-01  6.164720e-01  7.008000e-01  4.245360e-01, 0         IG01
1         IG02
2         MG01
3         MG02
4      Mysql01
        ...   
20    Tomcat02
21    Tomcat03
22    Tomcat04
23    apache01
24    apache02
Length: 25, dtype: object,    cmdb_id                          KPI_name  count       min         p50         p95         max        mean
0     IG01          trace.self.duration_mean     31  0.021500    0.105347    0.538222    0.593073    0.175682
1     IG01           trace.self.duration_p95     31  0.026000    0.781750    2.874875    3.886000    1.018566
2     IG01              trace.self.row_count     31  4.000000  446.000000  594.000000  736.000000  418.806452
3     IG01   trace.to_Tomcat01.duration_mean     31  0.016000    0.104125    0.368762    0.826289    0.137503
4     IG01    trace.to_Tomcat01.duration_p95     31  0.016000    0.472500    1.708300    3.188100    0.696094
5     IG01       trace.to_Tomcat01.row_count     31  1.000000   56.000000   74.500000   81.000000   51.387097
6     IG01   trace.to_Tomcat02.duration_mean     31  0.024000    0.089188    0.322577    0.682248    0.127612
7     IG01    trace.to_Tomcat02.duration_p95     31  0.024000    0.547000    2.116375    4.104000    0.746671
8     IG01       trace.to_Tomcat02.row_count     31  1.000000   52.000000   78.500000  101.000000   52.064516
9     IG02              trace.self.row_count     31  4.000000  442.000000  589.000000  758.000000  415.612903
10    IG02   trace.to_Tomcat01.duration_mean     31  0.027037    0.099656    0.376746    0.688147    0.139090
11    IG02           trace.self.duration_p95     31  0.049000    0.896750    2.836025    4.102000    1.048126
12    IG02          trace.self.duration_mean     31  0.034000    0.104258    0.497745    0.756012    0.178150
13    MG01     trace.from_Tomcat01.row_count     31  2.000000   55.000000   76.500000   90.000000   52.774194
14    IG02       trace.to_Tomcat02.row_count     31  1.000000   56.000000   82.000000   98.000000   53.000000
15    IG02    trace.to_Tomcat02.duration_p95     31  0.017000    0.760300    1.719300    4.111700    0.774482
16    IG02   trace.to_Tomcat02.duration_mean     31  0.017000    0.109526    0.365716    0.837609    0.138519
17    IG02       trace.to_Tomcat01.row_count     31  1.000000   54.000000   79.000000   97.000000   51.387097
18    IG02    trace.to_Tomcat01.duration_p95     31  0.048000    0.857300    1.699600    3.639850    0.799235
19    MG01  trace.from_Tomcat01.duration_p95     31  0.041000    0.174200    1.898100    3.882450    0.536556, 0         IG01
1         IG02
2         MG01
3         MG02
4     Tomcat01
5     Tomcat02
6     Tomcat03
7     Tomcat04
8     dockerA1
9     dockerA2
10    dockerB1
11    dockerB2
dtype: object,      cmdb_id          KPI_name  count    min      p50       p95     max        mean
0   Tomcat01     log.row_count     31   2.00  113.000  144.5000  172.00  102.967742
1   Tomcat01   log.error_count     31   0.00    0.000    0.0000    0.00    0.000000
2   apache01   log.error_count     31   0.00    0.000    0.0000    0.00    0.000000
3   apache01     log.row_count     31   1.00  217.000  295.0000  366.00  205.709677
4   Tomcat02   log.error_count     31   0.00    0.000    0.0000    0.00    0.000000
5   Tomcat02     log.row_count     31   2.00  108.000  159.5000  199.00  105.258065
6   apache02   log.error_count     31   0.00    0.000    0.0000    0.00    0.000000
7   apache02     log.row_count     31   3.00  222.000  295.5000  381.00  211.516129
8   Tomcat04   log.error_count     30   0.00    0.000    0.0000    0.00    0.000000
9   Tomcat04     log.row_count     30  57.00  108.500  152.7500  188.00  107.066667
10  Tomcat03     log.row_count     30  69.00  110.500  152.4000  190.00  109.733333
11  Tomcat03   log.error_count     30   0.00    0.000    0.0000    0.00    0.000000
12  Tomcat04  log.gc_pause_max      7   0.03    0.060    0.0940    0.10    0.061429
13  Tomcat02  log.gc_pause_max      6   0.04    0.065    0.1025    0.11    0.068333
14  Tomcat03  log.gc_pause_max      5   0.05    0.070    0.1440    0.16    0.084000
15  Tomcat01  log.gc_pause_max      5   0.03    0.060    0.0880    0.09    0.062000
16   Mysql02     log.row_count      2   1.00    1.500    1.9500    2.00    1.500000
17   Mysql02   log.error_count      2   0.00    0.000    0.0000    0.00    0.000000, 0     Mysql02
1    Tomcat01
2    Tomcat02
3    Tomcat03
4    Tomcat04
5    apache01
6    apache02
dtype: object, Empty DataFrame
Columns: [cmdb_id, count, earliest_timestamp_Asia/Shanghai]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing loaded dataframes if present; otherwise load from CSV
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

# Standardize KPI_name column
if 'kpi_name' in df_metric.columns:
    df_metric = df_metric.rename(columns={'kpi_name': 'KPI_name'})
if 'trace_name' in df_trace.columns:
    df_trace = df_trace.rename(columns={'trace_name': 'KPI_name'})
if 'log_name' in df_log.columns:
    df_log = df_log.rename(columns={'log_name': 'KPI_name'})

# Ensure value numeric
for df in (df_metric, df_trace, df_log):
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    # create timezone-aware timestamps columns (UTC->Asia/Shanghai)
    df['ts_utc'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    df['ts_sh'] = df['ts_utc'].dt.tz_convert(tz)

# Define incident window in Asia/Shanghai tz
window_start = tz.localize(pd.to_datetime("2021-03-23 00:00:00"))
window_end = tz.localize(pd.to_datetime("2021-03-23 00:30:00"))

def compute_anomalies(df, label):
    # 1) compute global p95 and p5 per group using the entire series (no filtering)
    thresholds = df.groupby(['cmdb_id','KPI_name'], as_index=False)['value'].agg(
        global_p95 = lambda x: x.quantile(0.95),
        global_p5  = lambda x: x.quantile(0.05),
        pts = 'size'
    )
    # 2) filter original file to the incident window using ts_sh
    df_window = df[(df['ts_sh'] >= window_start) & (df['ts_sh'] <= window_end)].copy()
    # 3) join thresholds into windowed data to mark anomalies
    df_w = df_window.merge(thresholds, on=['cmdb_id','KPI_name'], how='left')
    # mark anomalies
    df_w['is_high'] = df_w['value'] >= df_w['global_p95']
    df_w['is_low']  = df_w['value'] <= df_w['global_p5']
    df_w['is_anom'] = df_w['is_high'] | df_w['is_low']
    # store full anomaly rows table (all window rows with anomaly flag)
    anomalies_full = df_w.copy()
    # 3b) group within window to produce per-group summary
    grp_window = df_w.groupby(['cmdb_id','KPI_name'], as_index=False).agg(
        anomaly_count = ('is_anom','sum'),
        earliest_anom_ts = ('ts_sh', lambda s: s[s.index[df_w.loc[s.index,'is_anom']].min()] if (df_w.loc[s.index,'is_anom'].any()) else pd.NaT),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min'),
        any_high = ('is_high','any'),
        any_low  = ('is_low','any'),
        global_p95 = ('global_p95','first'),
        global_p5  = ('global_p5','first')
    )
    # Note: the earliest_anom_ts lambda above is a bit complex; simpler approach below to compute earliest_anom_ts correctly.
    # Recompute earliest_anom_ts reliably:
    if not df_w.empty:
        anom_times = df_w[df_w['is_anom']].groupby(['cmdb_id','KPI_name'], as_index=False)['ts_sh'].min().rename(columns={'ts_sh':'earliest_anom_ts'})
        grp_window = grp_window.drop(columns=['earliest_anom_ts']).merge(anom_times, on=['cmdb_id','KPI_name'], how='left')
    else:
        grp_window['earliest_anom_ts'] = pd.NaT

    # severity_ratio logic
    def compute_sev(row):
        gp95 = row['global_p95']
        gp5  = row['global_p5']
        maxv = row['max_value_in_window']
        minv = row['min_value_in_window']
        sev_candidates = []
        if row['any_high'] and pd.notnull(gp95) and gp95 != 0 and pd.notnull(maxv):
            sev_candidates.append(maxv / gp95)
        if row['any_low'] and pd.notnull(gp5) and pd.notnull(minv) and minv != 0:
            sev_candidates.append(gp5 / minv)
        if len(sev_candidates)==0:
            return np.nan
        return float(np.max(sev_candidates))
    grp_window['severity_ratio'] = grp_window.apply(compute_sev, axis=1)

    # For compactness, format earliest_anom_ts as string in Asia/Shanghai or NaN
    grp_window['earliest_anom_ts'] = grp_window['earliest_anom_ts'].dt.strftime("%Y-%m-%d %H:%M:%S%z")
    grp_window = grp_window.rename(columns={'earliest_anom_ts':'earliest_anomaly_timestamp'})

    # Replace NaT strings with NaN
    grp_window['earliest_anomaly_timestamp'] = grp_window['earliest_anomaly_timestamp'].replace({pd.NaT: np.nan, None: np.nan})

    # Round numeric columns for neatness
    for c in ['max_value_in_window','min_value_in_window','global_p95','global_p5','severity_ratio']:
        if c in grp_window.columns:
            grp_window[c] = grp_window[c].round(6)

    # Ensure all groups from thresholds are present in summary (even if they had no points in window)
    # Merge thresholds to include groups with no window points
    all_groups = thresholds[['cmdb_id','KPI_name','global_p95','global_p5']].copy()
    summary = all_groups.merge(grp_window, on=['cmdb_id','KPI_name','global_p95','global_p5'], how='left')
    # Fill missing numeric stats for groups with no data in window
    summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)
    summary['max_value_in_window'] = summary['max_value_in_window']
    summary['min_value_in_window'] = summary['min_value_in_window']
    summary['earliest_anomaly_timestamp'] = summary['earliest_anomaly_timestamp']
    summary['severity_ratio'] = summary['severity_ratio']

    # total anomaly points in file window
    total_anomalies = int(df_w['is_anom'].sum())

    # list of groups with anomaly_count>0 as "cmdb_id|KPI_name"
    groups_with_anoms = summary[summary['anomaly_count']>0].copy()
    groups_with_anoms['group_name'] = groups_with_anoms['cmdb_id'] + "|" + groups_with_anoms['KPI_name']
    groups_with_anoms_list = groups_with_anoms['group_name'].reset_index(drop=True)

    # top 20 groups sorted by anomaly_count desc (select specified columns)
    top20 = summary.sort_values('anomaly_count', ascending=False).head(20)
    top20 = top20[[
        'cmdb_id','KPI_name','anomaly_count','earliest_anomaly_timestamp',
        'max_value_in_window','min_value_in_window','global_p95','global_p5','severity_ratio'
    ]]

    # store full tables in kernel variables named by label
    globals()[f"{label}_thresholds"] = thresholds
    globals()[f"{label}_window"] = df_w
    globals()[f"{label}_anomalies_full"] = anomalies_full
    globals()[f"{label}_summary_full"] = summary
    globals()[f"{label}_summary_top20"] = top20
    globals()[f"{label}_total_anomalies"] = total_anomalies
    globals()[f"{label}_groups_with_anoms"] = groups_with_anoms_list

    return top20, total_anomalies, groups_with_anoms_list

# Process each file
metric_summary_top20, metric_total_anomalies, metric_groups_with_anomalies = compute_anomalies(df_metric, 'metric')
trace_summary_top20, trace_total_anomalies, trace_groups_with_anomalies   = compute_anomalies(df_trace, 'trace')
log_summary_top20, log_total_anomalies, log_groups_with_anomalies         = compute_anomalies(df_log, 'log')

# Present compact outputs (top20 summaries and totals and short group lists)
metric_summary_top20, metric_total_anomalies, metric_groups_with_anomalies.head(50), \
trace_summary_top20, trace_total_anomalies, trace_groups_with_anomalies.head(50), \
log_summary_top20, log_total_anomalies, log_groups_with_anomalies.head(50)

```
Out[2]:
```
Summary of anomaly detection results (incident window 2021-03-23 00:00–00:30 Asia/Shanghai):

1) metric.csv
- Total anomaly points in the window: 21,187.
- Top groups by anomaly_count (examples):
  - IG01 | JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed — anomaly_count=31, earliest=2021-03-23 00:00:00+0800, max=58,056,832.0, global_p95=58,056,832.0, severity_ratio=1.0
  - Many Mysql02 metrics (e.g., Mysql-MySQL_3306_Select Range, Table open cache overflows, Innodb pages read, etc.) show anomaly_count=31 with earliest=2021-03-23 00:00:00+0800.
- Number of distinct groups with anomaly_count>0 in the reported short list: 50 (examples shown above). Many groups had anomaly_count equal to the full window (31 points), indicating values at/above p95 or at/below p5 for the entire window.

2) trace.csv
- Total anomaly points in the window: 989.
- Top groups by anomaly_count (examples):
  - dockerA2 | trace.self.duration_p95 — anomaly_count=20, earliest=2021-03-23 00:01:00+0800, max=0.009, global_p95=0.0085, severity_ratio≈1.333
  - dockerA1 | trace.self.duration_p95 — anomaly_count=11, earliest=2021-03-23 00:00:00+0800, severity_ratio=1.0
  - Several IG01/IG02 and MG01 groups (trace row_count and duration metrics) with anomaly_count between 4 and 5 and severity ratios >1 for P95 breaches.
- Number of distinct groups with anomaly_count>0 in the short list: 50 (examples above). Earliest anomalies are at 00:00–00:02+0800.

3) log.csv
- Total anomaly points in the window: 221.
- Top groups by anomaly_count (examples):
  - Tomcat02 | log.error_count — anomaly_count=31, earliest=2021-03-23 00:00:00+0800, max=0.0, global_p95=0.0 (severity_ratio undefined)
  - Tomcat01 | log.error_count — anomaly_count=31, similar situation (0 values vs global thresholds 0)
  - apache01 | log.row_count — anomaly_count=4, earliest=2021-03-23 00:01:00+0800, max=366.0, global_p95=295.0, severity_ratio≈126.5/295? (severity shown in table)
- Many log.error_count groups show anomaly_count=30–31 but global_p95/global_p5 are 0, so severity_ratio is NaN (division undefined) — these are flagged because the values equal threshold (zeros).
- Short list of groups with anomalies contains ~13 named groups in the displayed sample (examples above).

Notes / interpretation
- Global p95/p5 were computed over the entire available series per (cmdb_id, KPI_name) before applying the time filter.
- Many metric groups (notably MySQL and IG01 JVM memory metrics) show anomalies for every sampled minute in the window (anomaly_count=31) — these require inspection as they either constant-hit p95/p5 or thresholds coincide with observed values.
- For some groups (especially log.error_count) global thresholds are zero; anomalies are detected because values match those thresholds but severity ratio is not defined (NaN) — treat these as presence-of-errors flags rather than magnitude breaches.
- All earliest anomaly timestamps are within the requested window and mostly at 2021-03-23 00:00 or 00:01+0800.

Stored kernel variables for further inspection
- metric_thresholds, metric_window, metric_anomalies_full, metric_summary_full, metric_summary_top20, metric_total_anomalies, metric_groups_with_anoms
- trace_thresholds, trace_window, trace_anomalies_full, trace_summary_full, trace_summary_top20, trace_total_anomalies, trace_groups_with_anoms
- log_thresholds, log_window, log_anomalies_full, log_summary_full, log_summary_top20, log_total_anomalies, log_groups_with_anoms

If you want, I can: (a) list the top 20 rows for any file's summary table, (b) export the specific groups with anomaly_count>0, or (c) drill into a particular cmdb_id/KPI_name for raw window points.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                        KPI_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  global_p95   global_p5  severity_ratio
3       IG01     JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             31   2021-03-23 00:00:00+0800           58056832.0           58056832.0  58056832.0  58056832.0        1.000000
726  Mysql02     Mysql-MySQL_3306_Table open cache overflows             31   2021-03-23 00:00:00+0800                  0.0                  0.0         0.0         0.0             NaN
727  Mysql02          Mysql-MySQL_3306_Tc log max pages used             31   2021-03-23 00:00:00+0800                  0.0                  0.0         0.0         0.0             NaN
712  Mysql02                   Mysql-MySQL_3306_Select Range             31   2021-03-23 00:00:00+0800                 13.0                  0.0        13.0         0.0        1.000000
713  Mysql02             Mysql-MySQL_3306_Select Range Check             31   2021-03-23 00:00:00+0800                  0.0                  0.0         0.0         0.0             NaN
715  Mysql02         Mysql-MySQL_3306_Slave Open Temp Tables             31   2021-03-23 00:00:00+0800                  0.0                  0.0         0.0         0.0             NaN
716  Mysql02            Mysql-MySQL_3306_Slow launch threads             31   2021-03-23 00:00:00+0800                  0.0                  0.0         0.0         0.0             NaN
718  Mysql02              Mysql-MySQL_3306_Sort Merge Passes             31   2021-03-23 00:00:00+0800                  6.0                  0.0         6.0         0.0        1.000000
729  Mysql02                 Mysql-MySQL_3306_Threads Cached             31   2021-03-23 00:00:00+0800                 15.0                 14.0        15.0        14.5        1.035714
730  Mysql02                Mysql-MySQL_3306_Threads Created             31   2021-03-23 00:00:00+0800                 37.0                  0.0        37.0         0.0        1.000000
731  Mysql02               Mysql-MySQL_3306_ThreadsConnected             31   2021-03-23 00:00:00+0800                 23.0                 22.0        22.5        22.0        1.022222
733  Mysql02      Mysql-MySQL_3306_max trx lock memory bytes             31   2021-03-23 00:00:00+0800               1136.0                  0.0      1136.0         0.0        1.000000
721  Mysql02                      Mysql-MySQL_3306_Sort Scan             31   2021-03-23 00:00:00+0800              21461.0                  0.0     21461.0         0.0        1.000000
723  Mysql02             Mysql-MySQL_3306_Table Locks Waited             31   2021-03-23 00:00:00+0800                  0.0                  0.0         0.0         0.0             NaN
681  Mysql02              Mysql-MySQL_3306_Innodb pages read             31   2021-03-23 00:00:00+0800            2024680.0                  0.0   2024680.0         0.0        1.000000
683  Mysql02  Mysql-MySQL_3306_Innodb row lock current waits             31   2021-03-23 00:00:00+0800                  0.0                  0.0         0.0         0.0             NaN
684  Mysql02       Mysql-MySQL_3306_Innodb row lock time avg             31   2021-03-23 00:00:00+0800                 16.0                 16.0        16.0        16.0        1.000000
685  Mysql02       Mysql-MySQL_3306_Innodb row lock time max             31   2021-03-23 00:00:00+0800               9522.0               9522.0      9522.0      9522.0        1.000000
686  Mysql02              Mysql-MySQL_3306_Key Read Requests             31   2021-03-23 00:00:00+0800               2790.0                  0.0      2790.0         0.0        1.000000
687  Mysql02                      Mysql-MySQL_3306_Key Reads             31   2021-03-23 00:00:00+0800                  7.0                  0.0         7.0         0.0        1.000000, 21187, 0         IG01|JVM-Memory_7778_JVM_Memory_HeapMemoryMax
1        IG01|JVM-Memory_7778_JVM_Memory_HeapMemoryUsed
2      IG01|JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed
3                  IG01|JVM-Runtime_7778_JVM_JVM_Uptime
4         IG01|JVM-Threads_7778_JVM_ThreadCount_Threads
                            ...                        
45    IG01|OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_D...
46    IG01|OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_D...
47          IG01|OSLinux-OSLinux_MEMORY_MEMORY_CacheMem
48        IG01|OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem
49    IG01|OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc
Name: group_name, Length: 50, dtype: object,       cmdb_id                          KPI_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  global_p95   global_p5  severity_ratio
202  dockerA2           trace.self.duration_p95             20   2021-03-23 00:01:00+0800             0.009000             0.000750    0.008500    0.001000        1.333333
187  dockerA1           trace.self.duration_p95             11   2021-03-23 00:00:00+0800             0.009000             0.001000    0.009000    0.001000        1.000000
104      MG02       trace.to_dockerA2.row_count              5   2021-03-23 00:02:00+0800            71.000000             1.000000   64.000000   22.500000       22.500000
200  dockerA2         trace.from_MG02.row_count              5   2021-03-23 00:02:00+0800            71.000000             1.000000   64.000000   22.500000       22.500000
182  dockerA1         trace.from_MG01.row_count              5   2021-03-23 00:01:00+0800            62.000000             1.000000   59.000000   19.000000       19.000000
62       MG01       trace.to_dockerA1.row_count              5   2021-03-23 00:01:00+0800            62.000000             1.000000   59.000000   19.000000       19.000000
239  dockerB2           trace.to_MG01.row_count              5   2021-03-23 00:01:00+0800           118.000000             3.000000   84.550000   18.000000        6.000000
53       MG01     trace.from_dockerB2.row_count              5   2021-03-23 00:01:00+0800           118.000000             3.000000   84.550000   18.000000        6.000000
8        IG01       trace.to_Tomcat02.row_count              4   2021-03-23 00:01:00+0800           101.000000             1.000000   78.500000   31.500000       31.500000
6        IG01   trace.to_Tomcat02.duration_mean              4   2021-03-23 00:00:00+0800             0.682248             0.024000    0.322577    0.030823        2.114993
17       IG02              trace.self.row_count              4   2021-03-23 00:00:00+0800           758.000000             4.000000  589.000000  252.000000       63.000000
32       MG01     trace.from_Tomcat01.row_count              4   2021-03-23 00:01:00+0800            90.000000             2.000000   76.500000   30.500000       15.250000
31       MG01  trace.from_Tomcat01.duration_p95              4   2021-03-23 00:00:00+0800             3.882450             0.041000    1.898100    0.042475        2.045440
38       MG01     trace.from_Tomcat03.row_count              4   2021-03-23 00:01:00+0800            95.000000            31.000000   75.100000   35.450000        1.264980
7        IG01    trace.to_Tomcat02.duration_p95              4   2021-03-23 00:00:00+0800             4.104000             0.024000    2.116375    0.054675        2.278125
15       IG02          trace.self.duration_mean              4   2021-03-23 00:00:00+0800             0.756012             0.034000    0.497745    0.048872        1.518874
23       IG02       trace.to_Tomcat02.row_count              4   2021-03-23 00:00:00+0800            98.000000             1.000000   82.000000   26.000000       26.000000
22       IG02    trace.to_Tomcat02.duration_p95              4   2021-03-23 00:00:00+0800             4.111700             0.017000    1.719300    0.040900        2.405882
20       IG02       trace.to_Tomcat01.row_count              4   2021-03-23 00:01:00+0800            97.000000             1.000000   79.000000   28.000000       28.000000
60       MG01   trace.to_dockerA1.duration_mean              4   2021-03-23 00:00:00+0800             0.180048             0.022132    0.111672    0.022342        1.612293, 989, 0                  IG01|trace.self.row_count
1        IG01|trace.to_Tomcat01.duration_p95
2           IG01|trace.to_Tomcat01.row_count
3       IG01|trace.to_Tomcat02.duration_mean
4        IG01|trace.to_Tomcat02.duration_p95
                       ...                  
45        Tomcat02|trace.from_IG02.row_count
46             Tomcat02|trace.self.row_count
47          Tomcat02|trace.to_MG02.row_count
48    Tomcat03|trace.from_root.duration_mean
49     Tomcat03|trace.from_root.duration_p95
Name: group_name, Length: 50, dtype: object,      cmdb_id          KPI_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  global_p95  global_p5  severity_ratio
5   Tomcat02   log.error_count             31   2021-03-23 00:00:00+0800                 0.00                 0.00      0.0000     0.0000             NaN
2   Tomcat01   log.error_count             31   2021-03-23 00:00:00+0800                 0.00                 0.00      0.0000     0.0000             NaN
14  apache01   log.error_count             31   2021-03-23 00:00:00+0800                 0.00                 0.00      0.0000     0.0000             NaN
16  apache02   log.error_count             31   2021-03-23 00:00:00+0800                 0.00                 0.00      0.0000     0.0000             NaN
8   Tomcat03   log.error_count             30   2021-03-23 00:00:00+0800                 0.00                 0.00      0.0000     0.0000             NaN
11  Tomcat04   log.error_count             30   2021-03-23 00:00:00+0800                 0.00                 0.00      0.0000     0.0000             NaN
15  apache01     log.row_count              4   2021-03-23 00:01:00+0800               366.00                 1.00    295.0000   126.5000      126.500000
17  apache02     log.row_count              4   2021-03-23 00:00:00+0800               381.00                 3.00    295.5000   132.0000       44.000000
7   Tomcat02     log.row_count              4   2021-03-23 00:00:00+0800               199.00                 2.00    159.5000    60.5000       30.250000
4   Tomcat01     log.row_count              4   2021-03-23 00:01:00+0800               172.00                 2.00    144.5000    59.0000       29.500000
13  Tomcat04     log.row_count              4   2021-03-23 00:01:00+0800               188.00                57.00    152.7500    60.3500        1.230769
0    Mysql02   log.error_count              2   2021-03-23 00:00:00+0800                 0.00                 0.00      0.0000     0.0000             NaN
1    Mysql02     log.row_count              2   2021-03-23 00:00:00+0800                 2.00                 1.00      1.9500     1.0500        1.050000
3   Tomcat01  log.gc_pause_max              2   2021-03-23 00:10:00+0800                 0.09                 0.03      0.0880     0.0340        1.133333
9   Tomcat03  log.gc_pause_max              0                        NaN                  NaN                  NaN      0.1440     0.0520             NaN
6   Tomcat02  log.gc_pause_max              0                        NaN                  NaN                  NaN      0.1025     0.0425             NaN
12  Tomcat04  log.gc_pause_max              0                        NaN                  NaN                  NaN      0.0940     0.0300             NaN
10  Tomcat03     log.row_count              0                        NaN                  NaN                  NaN    152.4000    71.8000             NaN, 221, 0       Mysql02|log.error_count
1         Mysql02|log.row_count
2      Tomcat01|log.error_count
3     Tomcat01|log.gc_pause_max
4        Tomcat01|log.row_count
5      Tomcat02|log.error_count
6        Tomcat02|log.row_count
7      Tomcat03|log.error_count
8      Tomcat04|log.error_count
9        Tomcat04|log.row_count
10     apache01|log.error_count
11       apache01|log.row_count
12     apache02|log.error_count
13       apache02|log.row_count
Name: group_name, dtype: object)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: metric_window, metric_thresholds
if 'metric_window' not in globals() or 'metric_thresholds' not in globals():
    raise RuntimeError("Required variables metric_window and metric_thresholds not found in kernel.")

tz = pytz.timezone('Asia/Shanghai')

# Ensure necessary columns present in metric_window; if thresholds not merged, merge them
if 'global_p95' not in metric_window.columns or 'global_p5' not in metric_window.columns:
    metric_window = metric_window.merge(metric_thresholds[['cmdb_id','KPI_name','global_p95','global_p5']],
                                        on=['cmdb_id','KPI_name'], how='left')

# Ensure types
metric_window = metric_window.copy()
metric_window['value'] = pd.to_numeric(metric_window['value'], errors='coerce')
metric_window['global_p95'] = pd.to_numeric(metric_window['global_p95'], errors='coerce')
metric_window['global_p5']  = pd.to_numeric(metric_window['global_p5'], errors='coerce')

# Recompute is_anom flag to be safe
metric_window['is_anom'] = ((metric_window['value'] >= metric_window['global_p95']) |
                            (metric_window['value'] <= metric_window['global_p5']))

# Prepare containers
rows = []
per_point_dfs = []

# Function to process each group
def process_group(g):
    g = g.sort_values('ts_sh').copy()
    # compute time gap in seconds
    ts_diff = g['ts_sh'].diff().dt.total_seconds().fillna(999)
    gap_flag = ts_diff != 60  # True if not consecutive minute
    prev_anom = g['is_anom'].shift(1).fillna(False)
    # start of a new anomalous run
    start = g['is_anom'] & ( (~prev_anom) | gap_flag )
    # run id: cumulative sum of starts (per-group)
    run_id = start.cumsum()
    # set run_id to NaN for non-anomalous rows
    run_id = run_id.where(g['is_anom'], np.nan)
    g['run_id'] = run_id
    per_point_dfs.append(g)

    total_anom_points = int(g['is_anom'].sum())

    # compute runs summary
    runs = g[g['is_anom']].copy()
    if runs.empty:
        return {
            'cmdb_id': g['cmdb_id'].iloc[0],
            'KPI_name': g['KPI_name'].iloc[0],
            'total_anomaly_points': total_anom_points,
            'max_consecutive_run_length': 0,
            'first_run_start_timestamp': pd.NaT,
            'first_run_end_timestamp': pd.NaT,
            'first_run_max_value': np.nan,
            'global_p95': g['global_p95'].iloc[0],
            'global_p5' : g['global_p5'].iloc[0]
        }
    run_groups = runs.groupby('run_id', dropna=True)
    run_stats = run_groups.agg(
        run_len = ('run_id','size'),
        run_start = ('ts_sh','min'),
        run_end   = ('ts_sh','max'),
        run_max_value = ('value','max')
    ).reset_index()
    # find max length
    max_len = run_stats['run_len'].max()
    # pick the first run with that length (earliest start)
    cand = run_stats[run_stats['run_len']==max_len].sort_values('run_start').iloc[0]
    return {
        'cmdb_id': g['cmdb_id'].iloc[0],
        'KPI_name': g['KPI_name'].iloc[0],
        'total_anomaly_points': total_anom_points,
        'max_consecutive_run_length': int(cand['run_len']),
        'first_run_start_timestamp': cand['run_start'],
        'first_run_end_timestamp': cand['run_end'],
        'first_run_max_value': float(cand['run_max_value']),
        'global_p95': g['global_p95'].iloc[0],
        'global_p5' : g['global_p5'].iloc[0]
    }

# Apply per-group processing
grouped = metric_window.groupby(['cmdb_id','KPI_name'], sort=False)
summary_list = []
for _, g in grouped:
    summary_list.append(process_group(g))

metric_runs_summary_full = pd.DataFrame(summary_list)

# Filter to groups with max_consecutive_run_length >= 3
metric_runs_filtered = metric_runs_summary_full[metric_runs_summary_full['max_consecutive_run_length'] >= 3].copy()

# Format timestamps as Asia/Shanghai strings
def fmt_ts(x):
    if pd.isna(x):
        return np.nan
    # ensure tz-aware and format
    if isinstance(x, pd.Timestamp) and x.tzinfo is not None:
        return x.strftime("%Y-%m-%d %H:%M:%S%z")
    # try to localize if naive (shouldn't be necessary)
    try:
        return pd.to_datetime(x).tz_localize(tz).strftime("%Y-%m-%d %H:%M:%S%z")
    except Exception:
        return np.nan

metric_runs_filtered['first_run_start_timestamp'] = metric_runs_filtered['first_run_start_timestamp'].apply(fmt_ts)
metric_runs_filtered['first_run_end_timestamp']   = metric_runs_filtered['first_run_end_timestamp'].apply(fmt_ts)

# Round numeric columns
for c in ['first_run_max_value','global_p95','global_p5']:
    if c in metric_runs_filtered.columns:
        metric_runs_filtered[c] = pd.to_numeric(metric_runs_filtered[c], errors='coerce').round(6)

# Sort by max_consecutive_run_length desc then total_anomaly_points desc, take top 20
metric_runs_top20 = metric_runs_filtered.sort_values(
    ['max_consecutive_run_length','total_anomaly_points'],
    ascending=[False, False]
).head(20).reset_index(drop=True)

# Distinct cmdb_id values in these groups (limit 20)
metric_runs_cmdbs = pd.Series(sorted(metric_runs_top20['cmdb_id'].dropna().unique()))[:20]

# Store full per-point run detection results in kernel variable
metric_run_points = pd.concat(per_point_dfs, ignore_index=True) if per_point_dfs else pd.DataFrame()

# Final compact outputs
metric_runs_top20, metric_runs_cmdbs

```
Out[3]:
```
```python
# Plain-English summary string of the run-detection results (displayed as a single variable)
summary = (
    "Fault identification summary (metric data, window 2021-03-23 00:00–00:30 Asia/Shanghai):\n\n"
    "• Detection method: points labeled anomalous when value >= group's global_p95 or <= global_p5 "
    "(global p5/p95 computed from each group's full series). Consecutive-anomaly runs are consecutive-minute "
    "anomalous points; we kept groups with max run length >= 3 minutes.\n\n"
    "• Top findings (compact):\n"
    "  - Many groups show a continuous anomalous run spanning the entire 31-minute window (max_consecutive_run_length = 31).\n"
    "  - Example top groups (each has total_anomaly_points=31 and run start 2021-03-23 00:00:00+0800 -> end 00:30:00+0800):\n"
    "      * IG01 | JVM-Memory_7778_JVM_Memory_HeapMemoryMax (first_run_max_value ≈ 2.040136e+09, global_p95 = 2.040136e+09)\n"
    "      * IG01 | JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed (first_run_max_value ≈ 5.805683e+07)\n"
    "      * IG02 | JVM-Memory_7778_JVM_Memory_HeapMemoryMax (same full-window run)\n"
    "      * MG01 / MG02 similar JVM memory metrics and several OSLinux network/process metrics (many with 31-minute runs).\n"
    "      * Mysql02 shows many MySQL-related metrics with full-window anomalous runs (e.g., Aborted Clients, Binlog uses, Innodb reads).\n\n"
    "• Interpretation: A max_consecutive_run_length of 31 means the metric stayed at or beyond the group's global threshold "
    "for the entire incident window — not isolated spikes but sustained deviations. This appears across JVM memory KPIs, "
    "numerous MySQL KPIs, and some OS-network/process KPIs for hosts IG01, IG02, MG01, MG02, and Mysql02.\n\n"
    "• Distinct cmdb_id values present among the filtered groups (examples, limited to 20):\n"
    "  IG01, IG02, MG01, MG02, Mysql02\n\n"
    "• Stored kernel variables for later inspection:\n"
    "  - Per-point run detection table: metric_run_points\n"
    "  - Full per-group run summary: metric_runs_summary_full\n"
    "  - Filtered groups (max run >=3) and top-20 view: metric_runs_filtered, metric_runs_top20\n\n"
    "If you want, I can (a) expand the top-20 table with full values, (b) list all filtered groups, or (c) show raw per-minute "
    "point sequences for any specific group (cmdb_id + KPI_name)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                           KPI_name  total_anomaly_points  max_consecutive_run_length first_run_start_timestamp   first_run_end_timestamp  first_run_max_value    global_p95     global_p5
0      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         2.040136e+09  2.040136e+09  2.040136e+09
1      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         5.805683e+07  5.805683e+07  5.805683e+07
2      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         2.040136e+09  2.040136e+09  2.040136e+09
3      IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         5.811566e+07  5.811566e+07  5.811566e+07
4      MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         2.040136e+09  2.040136e+09  2.040136e+09
5      MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         5.894522e+07  5.894522e+07  5.894522e+07
6      MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00
7      MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         2.000000e+00  0.000000e+00  0.000000e+00
8      MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         2.040136e+09  2.040136e+09  2.040136e+09
9      MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00
10     MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00
11     MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         1.000000e+00  1.000000e+00  1.000000e+00
12     MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         4.000000e-01  3.500000e-01  3.000000e-01
13     MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         3.000000e+00  3.000000e+00  3.000000e+00
14  Mysql02                   Mysql-MySQL_3306_Aborted Clients                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         5.968000e+03  5.968000e+03  0.000000e+00
15  Mysql02                  Mysql-MySQL_3306_Aborted Connects                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         1.340000e+02  1.340000e+02  0.000000e+00
16  Mysql02             Mysql-MySQL_3306_Binlog Cache Disk Use                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         7.500000e+01  7.500000e+01  0.000000e+00
17  Mysql02        Mysql-MySQL_3306_Binlog stmt cache disk use                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00
18  Mysql02             Mysql-MySQL_3306_Binlog stmt cache use                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         4.321000e+03  4.321000e+03  4.312000e+03
19  Mysql02                        Mysql-MySQL_3306_Com Delete                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800         2.100000e+01  2.100000e+01  0.000000e+00, 0       IG01
1       IG02
2       MG01
3       MG02
4    Mysql02
dtype: object)```
```

In [ ]:
# In[4]:

import re
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing variables from kernel
if 'trace_summary_full' not in globals():
    raise RuntimeError("trace_summary_full not found in kernel.")
if 'metric_runs_filtered' not in globals():
    raise RuntimeError("metric_runs_filtered not found in kernel. (Needed for candidate cmdb_ids)")

# Work on a copy
trace_df = trace_summary_full.copy()

# Ensure consistent column name for the trace KPI
kpi_col = 'KPI_name' if 'KPI_name' in trace_df.columns else ('trace_name' if 'trace_name' in trace_df.columns else None)
if kpi_col is None:
    raise RuntimeError("Trace summary does not contain KPI_name/trace_name column.")

# Filter to groups with anomaly_count > 0
trace_positive = trace_df[trace_df['anomaly_count'] > 0].copy()

# Function to parse direction and peer from trace name
def parse_direction_peer(name, cmdb_id):
    # default
    direction = 'unknown'
    peer = None
    if pd.isna(name):
        return direction, peer
    s = str(name)
    # self
    if 'self' in s:
        return 'self', cmdb_id
    # look for to_<NAME> or from_<NAME>
    m_to = re.search(r'\bto_([A-Za-z0-9]+)', s)
    m_from = re.search(r'\bfrom_([A-Za-z0-9]+)', s)
    if m_to:
        return 'to', m_to.group(1)
    if m_from:
        return 'from', m_from.group(1)
    # fallback: if contains "to" or "from" words with different separators
    m_to2 = re.search(r'\bto([A-Za-z0-9]+)', s)
    m_from2 = re.search(r'\bfrom([A-Za-z0-9]+)', s)
    if m_to2:
        return 'to', m_to2.group(1)
    if m_from2:
        return 'from', m_from2.group(1)
    return 'unknown', None

# Apply parsing
trace_positive['direction_peer'] = trace_positive.apply(lambda r: parse_direction_peer(r[kpi_col], r['cmdb_id']), axis=1)
trace_positive[['direction','peer']] = pd.DataFrame(trace_positive['direction_peer'].tolist(), index=trace_positive.index)
trace_positive = trace_positive.drop(columns=['direction_peer'])

# Candidate cmdb_ids from metric fault groups (use metric_runs_filtered cmdb_id list)
candidate_cmdbs = set(metric_runs_filtered['cmdb_id'].unique())

# Keep rows where either trace row's cmdb_id OR parsed peer is in candidate list
def peer_in_candidates(peer):
    return (peer in candidate_cmdbs) if pd.notna(peer) else False

trace_filtered = trace_positive[
    (trace_positive['cmdb_id'].isin(candidate_cmdbs)) |
    (trace_positive['peer'].apply(peer_in_candidates))
].copy()

# Ensure earliest_anomaly_timestamp is datetime for sorting (it may be string); keep original string for display
if 'earliest_anomaly_timestamp' in trace_filtered.columns:
    trace_filtered['_earliest_dt'] = pd.to_datetime(trace_filtered['earliest_anomaly_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)
else:
    trace_filtered['_earliest_dt'] = pd.NaT

# Select and order columns for compact table
compact_cols = ['cmdb_id','direction','peer', kpi_col, 'anomaly_count','earliest_anomaly_timestamp','max_value_in_window','severity_ratio']
# Ensure columns exist; if not, add NaNs
for c in compact_cols:
    if c not in trace_filtered.columns:
        trace_filtered[c] = pd.NA

trace_filtered_compact = trace_filtered[compact_cols].copy()

# Sort by anomaly_count desc then earliest_anomaly_timestamp asc
trace_filtered_compact = trace_filtered_compact.assign(
    _sort_anom = pd.to_numeric(trace_filtered_compact['anomaly_count'], errors='coerce').fillna(0).astype(int),
    _sort_ts = pd.to_datetime(trace_filtered_compact['earliest_anomaly_timestamp'], utc=True, errors='coerce')
)
trace_filtered_compact = trace_filtered_compact.sort_values(by=['_sort_anom','_sort_ts'], ascending=[False, True]).drop(columns=['_sort_anom','_sort_ts'])

# Limit to top 20 rows for display
trace_filtered_top20 = trace_filtered_compact.head(20).reset_index(drop=True)

# Peer aggregate: peer name, count of trace-groups involving that peer, earliest trace anomaly timestamp for that peer
peer_agg = trace_filtered.groupby('peer', dropna=True).agg(
    trace_group_count = ('KPI_name','size') if kpi_col=='KPI_name' else (kpi_col,'size'),
    earliest_anom_ts = ('earliest_anomaly_timestamp','min')
).reset_index().rename(columns={'peer':'peer_name'})

# Parse earliest_anom_ts to datetime for sorting, keep string representation
peer_agg['_earliest_dt'] = pd.to_datetime(peer_agg['earliest_anom_ts'], utc=True, errors='coerce').dt.tz_convert(tz)
peer_agg_sorted = peer_agg.sort_values(by=['trace_group_count','_earliest_dt'], ascending=[False, True]).head(20).reset_index(drop=True)
peer_agg_display = peer_agg_sorted[['peer_name','trace_group_count','earliest_anom_ts']].rename(columns={'earliest_anom_ts':'earliest_anomaly_timestamp'})

# Store full filtered trace rows in kernel variable for later inspection
trace_filtered_candidates_full = trace_filtered.copy()

# Final compact outputs
trace_filtered_top20, peer_agg_display

```
Out[4]:
```
```python
# Build a plain-English summary from previously computed variables:
# trace_filtered_top20, peer_agg_display, trace_filtered_candidates_full

rows = []
rows.append("Trace-level fault summary (filtered traces with anomalies involving metric-fault candidates [IG01, IG02, MG01, MG02, Mysql02]):")
rows.append("")
rows.append("Top matched trace groups (examples, sorted by anomaly_count desc):")

# take up to 8 example rows from trace_filtered_top20
n_examples = min(8, len(trace_filtered_top20))
for i in range(n_examples):
    r = trace_filtered_top20.iloc[i]
    rows.append(f"  - {r['cmdb_id']} | {r['direction']} -> {r['peer']} | {r['KPI_name']} | anomalies={int(r['anomaly_count'])} | earliest={r['earliest_anomaly_timestamp']} | max={r['max_value_in_window']} | severity={r['severity_ratio']}")

rows.append("")
rows.append("Peer-level aggregation (top peers by number of involved trace groups):")
# list top 10 peers from peer_agg_display
n_peers = min(10, len(peer_agg_display))
for i in range(n_peers):
    p = peer_agg_display.iloc[i]
    rows.append(f"  - {p['peer_name']} | trace_groups={int(p['trace_group_count'])} | earliest_anomaly={p['earliest_anomaly_timestamp']}")

rows.append("")
rows.append("Interpretation / next steps:")
rows.append("  - Many anomalous trace groups involve MG01/MG02 and IG01/IG02, and peers like Tomcat01/Tomcat02 and dockerA/B.")
rows.append("  - These traces show anomalous counts or latency-related metrics (row_count, duration_p95/mean) during the incident window.")
rows.append("  - Recommend focusing on the peer hosts with highest involvement (root, MG01, Tomcat01, MG02, Tomcat02, IG01, IG02) for root-cause triage.")
rows.append("")
rows.append("Stored for inspection: trace_filtered_candidates_full (full filtered rows).")

summary = "\n".join(rows)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id direction      peer                          KPI_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  severity_ratio
0       MG01      from  dockerB2     trace.from_dockerB2.row_count              5   2021-03-23 00:01:00+0800           118.000000        6.000000
1       MG01        to  dockerA1       trace.to_dockerA1.row_count              5   2021-03-23 00:01:00+0800            62.000000       19.000000
2   dockerA1      from      MG01         trace.from_MG01.row_count              5   2021-03-23 00:01:00+0800            62.000000       19.000000
3   dockerB2        to      MG01           trace.to_MG01.row_count              5   2021-03-23 00:01:00+0800           118.000000        6.000000
4       MG02        to  dockerA2       trace.to_dockerA2.row_count              5   2021-03-23 00:02:00+0800            71.000000       22.500000
5   dockerA2      from      MG02         trace.from_MG02.row_count              5   2021-03-23 00:02:00+0800            71.000000       22.500000
6       IG01        to  Tomcat01    trace.to_Tomcat01.duration_p95              4   2021-03-23 00:00:00+0800             3.188100        2.918750
7       IG01        to  Tomcat02   trace.to_Tomcat02.duration_mean              4   2021-03-23 00:00:00+0800             0.682248        2.114993
8       IG01        to  Tomcat02    trace.to_Tomcat02.duration_p95              4   2021-03-23 00:00:00+0800             4.104000        2.278125
9       IG02      self      IG02          trace.self.duration_mean              4   2021-03-23 00:00:00+0800             0.756012        1.518874
10      IG02      self      IG02              trace.self.row_count              4   2021-03-23 00:00:00+0800           758.000000       63.000000
11      IG02        to  Tomcat02    trace.to_Tomcat02.duration_p95              4   2021-03-23 00:00:00+0800             4.111700        2.405882
12      IG02        to  Tomcat02       trace.to_Tomcat02.row_count              4   2021-03-23 00:00:00+0800            98.000000       26.000000
13      MG01      from  Tomcat01  trace.from_Tomcat01.duration_p95              4   2021-03-23 00:00:00+0800             3.882450        2.045440
14      MG01        to  dockerA1   trace.to_dockerA1.duration_mean              4   2021-03-23 00:00:00+0800             0.180048        1.612293
15      MG01        to  dockerA1    trace.to_dockerA1.duration_p95              4   2021-03-23 00:00:00+0800             0.999550        2.419632
16      MG01        to  dockerA2       trace.to_dockerA2.row_count              4   2021-03-23 00:00:00+0800            58.000000        2.040909
17      MG02      from  Tomcat02     trace.from_Tomcat02.row_count              4   2021-03-23 00:00:00+0800            96.000000       14.750000
18      MG02      self      MG02          trace.self.duration_mean              4   2021-03-23 00:00:00+0800             0.439189        1.706178
19      MG02      self      MG02           trace.self.duration_p95              4   2021-03-23 00:00:00+0800             3.363500        2.200000,    peer_name  trace_group_count earliest_anomaly_timestamp
0       root                 12   2021-03-23 00:00:00+0800
1       MG01                  9   2021-03-23 00:00:00+0800
2   Tomcat01                  8   2021-03-23 00:00:00+0800
3       MG02                  7   2021-03-23 00:00:00+0800
4   Tomcat02                  7   2021-03-23 00:00:00+0800
5       IG01                  6   2021-03-23 00:00:00+0800
6       IG02                  5   2021-03-23 00:00:00+0800
7   dockerA1                  3   2021-03-23 00:00:00+0800
8   dockerA2                  3   2021-03-23 00:00:00+0800
9   dockerB2                  2   2021-03-23 00:00:00+0800
10  Tomcat03                  1   2021-03-23 00:01:00+0800
11  Tomcat04                  1   2021-03-23 00:01:00+0800)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure required kernel variables exist
if 'metric_runs_filtered' not in globals():
    raise RuntimeError("metric_runs_filtered not found in kernel.")
if 'metric_runs_summary_full' not in globals():
    raise RuntimeError("metric_runs_summary_full not found in kernel.")

# Work on a copy
df = metric_runs_filtered.copy()

# Ensure necessary numeric columns exist
for col in ['first_run_max_value','global_p95','global_p5','max_consecutive_run_length','total_anomaly_points','KPI_name']:
    if col not in df.columns:
        df[col] = np.nan

# Parse start/end timestamps to tz-aware datetimes (they may be strings like "...+0800")
for col in ['first_run_start_timestamp','first_run_end_timestamp']:
    if col in df.columns:
        df[col + "_dt"] = pd.to_datetime(df[col], utc=True, errors='coerce').dt.tz_convert(tz)
    else:
        df[col + "_dt"] = pd.NaT

# Compute severity_ratio per group robustly using first_run_max_value and global thresholds
def compute_sev(row):
    maxv = pd.to_numeric(row.get('first_run_max_value', np.nan), errors='coerce')
    gp95 = pd.to_numeric(row.get('global_p95', np.nan), errors='coerce')
    gp5  = pd.to_numeric(row.get('global_p5', np.nan), errors='coerce')
    # Prefer p95-based severity if maxv >= gp95
    if pd.notnull(maxv) and pd.notnull(gp95) and gp95 != 0 and maxv >= gp95:
        return float(maxv / gp95)
    # Else if maxv <= gp5 (unlikely for max, but keep logic)
    if pd.notnull(maxv) and pd.notnull(gp5) and pd.notnull(maxv) and maxv != 0 and maxv <= gp5:
        return float(gp5 / maxv)
    # Fallback: if gp95 is available and non-zero, return ratio as maxv/gp95
    if pd.notnull(maxv) and pd.notnull(gp95) and gp95 != 0:
        return float(maxv / gp95)
    return np.nan

df['severity_ratio_calc'] = df.apply(compute_sev, axis=1)

# Prepare per-cmdb aggregation
grouped = df.groupby('cmdb_id', as_index=False)

metric_per_cmdb_full = grouped.agg(
    faulty_group_count = ('KPI_name', 'nunique'),
    earliest_fault_start_dt = ('first_run_start_timestamp_dt', 'min'),
    latest_fault_end_dt = ('first_run_end_timestamp_dt', 'max'),
    max_severity_ratio = ('severity_ratio_calc', 'max')
)

# Determine example_top_KPI per cmdb_id: highest severity_ratio, tie-breaker by max_consecutive_run_length then total_anomaly_points
df_for_example = df.sort_values(
    by=['cmdb_id','severity_ratio_calc','max_consecutive_run_length','total_anomaly_points'],
    ascending=[True, False, False, False]
).copy()

example_map = df_for_example.groupby('cmdb_id', as_index=False).first()[['cmdb_id','KPI_name']].rename(columns={'KPI_name':'example_top_KPI'})

metric_per_cmdb_full = metric_per_cmdb_full.merge(example_map, on='cmdb_id', how='left')

# Format timestamps as strings in Asia/Shanghai
metric_per_cmdb_full['earliest_fault_start_timestamp'] = metric_per_cmdb_full['earliest_fault_start_dt'].dt.strftime("%Y-%m-%d %H:%M:%S%z")
metric_per_cmdb_full['latest_fault_end_timestamp'] = metric_per_cmdb_full['latest_fault_end_dt'].dt.strftime("%Y-%m-%d %H:%M:%S%z")

# Select and order columns
metric_per_cmdb_full = metric_per_cmdb_full[[
    'cmdb_id',
    'faulty_group_count',
    'earliest_fault_start_timestamp',
    'latest_fault_end_timestamp',
    'max_severity_ratio',
    'example_top_KPI'
]]

# Round severity
metric_per_cmdb_full['max_severity_ratio'] = pd.to_numeric(metric_per_cmdb_full['max_severity_ratio'], errors='coerce').round(6)

# Sort by faulty_group_count desc and take top 20
metric_per_cmdb_top20 = metric_per_cmdb_full.sort_values(by='faulty_group_count', ascending=False).head(20).reset_index(drop=True)

# Store full grouping results for later inspection (variable already set as metric_per_cmdb_full)
# Final compact output
metric_per_cmdb_top20

```
Out[5]:
```
```python
# Build a plain-English summary string from the previously computed metric_per_cmdb_top20
rows = []
rows.append("Per-cmdb_id fault summary (top 20 by faulty_group_count) — incident window 2021-03-23 00:00–00:30 Asia/Shanghai:")
rows.append("")
rows.append("Top cmdb_id entries (cmdb_id | faulty_group_count | earliest_start -> latest_end | max_severity_ratio | example_top_KPI):")

# Use metric_per_cmdb_top20 from kernel
for _, r in metric_per_cmdb_top20.head(10).iterrows():
    rows.append(f" - {r['cmdb_id']} | {int(r['faulty_group_count'])} | {r['earliest_fault_start_timestamp']} -> {r['latest_fault_end_timestamp']} | max_severity={r['max_severity_ratio']} | example={r['example_top_KPI'][:80]}")

rows.append("")
rows.append("Key observations:")
rows.append(" - Mysql01 and Mysql02 are the most affected: Mysql01 has 125 faulty KPI groups, Mysql02 has 117. Their faults span the entire window (earliest start around 00:00–00:01 and latest end 00:30).")
rows.append(" - Several app/web/front-end nodes (Tomcat01–04) also show many faulty groups (24–27 groups) with higher severity ratios (Tomcat03 and Tomcat04 show severity up to 2.0).")
rows.append(" - Redis nodes (Redis02, Redis01) and MG/IG hosts (MG02, IG02, MG01, IG01) appear in the top list as well; many groups are sustained multi-minute faults (we filtered for runs >=3 minutes).")
rows.append(" - example_top_KPI shows the highest-severity KPI for each host (e.g., InnoDB buffer pool pages dirty for Mysql01/Mysql02; various JVM and OS/Network KPIs for others).")
rows.append("")
rows.append("Interpretation / next steps:")
rows.append(" - Prioritize Mysql01 and Mysql02 for triage given the large number of sustained faulty KPIs and elevated severity.")
rows.append(" - Also investigate Tomcat nodes (especially Tomcat03/04) and Redis nodes shown above because of high faulty_group_count or high severity ratios.")
rows.append(" - Use stored full grouping result (kernel variable: metric_per_cmdb_full) and per-group details (metric_runs_summary_full / metric_runs_filtered) to drill into specific KPIs and raw per-minute sequences.")
rows.append("")
summary = "\n".join(rows)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  faulty_group_count earliest_fault_start_timestamp latest_fault_end_timestamp  max_severity_ratio                                    example_top_KPI
0         Mysql01                 125       2021-03-23 00:01:00+0800   2021-03-23 00:30:00+0800            1.290323    Mysql-MySQL_3306_Innodb buffer pool pages dirty
1         Mysql02                 117       2021-03-23 00:00:00+0800   2021-03-23 00:30:00+0800            1.025795    Mysql-MySQL_3306_Innodb buffer pool pages dirty
2         Redis02                  36       2021-03-23 00:00:00+0800   2021-03-23 00:28:00+0800            1.156069                            OSLinux-CPU_CPU_CPULoad
3         Redis01                  29       2021-03-23 00:01:00+0800   2021-03-23 00:17:00+0800            1.000209           OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem
4        Tomcat04                  27       2021-03-23 00:00:00+0800   2021-03-23 00:30:00+0800            2.000000  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...
5        Tomcat02                  25       2021-03-23 00:00:00+0800   2021-03-23 00:30:00+0800            1.142857  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...
6        Tomcat03                  25       2021-03-23 00:00:00+0800   2021-03-23 00:30:00+0800            2.000000     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT
7        Tomcat01                  24       2021-03-23 00:00:00+0800   2021-03-23 00:30:00+0800            1.500000  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...
8        apache02                  13       2021-03-23 00:02:00+0800   2021-03-23 00:29:00+0800            1.000000       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc
9            MG02                  12       2021-03-23 00:00:00+0800   2021-03-23 00:30:00+0800            1.142857  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...
10           IG02                  11       2021-03-23 00:00:00+0800   2021-03-23 00:30:00+0800            1.290323  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...
11           MG01                  11       2021-03-23 00:00:00+0800   2021-03-23 00:30:00+0800            1.000000           JVM-Memory_7779_JVM_Memory_HeapMemoryMax
12           IG01                  11       2021-03-23 00:00:00+0800   2021-03-23 00:30:00+0800            1.000000           JVM-Memory_7778_JVM_Memory_HeapMemoryMax
13       apache01                  10       2021-03-23 00:03:00+0800   2021-03-23 00:30:00+0800            1.000000       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc
14   ServiceTest1                   2       2021-03-23 00:00:00+0800   2021-03-23 00:30:00+0800            1.000000                                             app.rr
15  ServiceTest10                   2       2021-03-23 00:00:00+0800   2021-03-23 00:30:00+0800            1.000000                                             app.rr
16  ServiceTest11                   2       2021-03-23 00:00:00+0800   2021-03-23 00:30:00+0800            1.000000                                             app.rr
17   ServiceTest7                   2       2021-03-23 00:00:00+0800   2021-03-23 00:30:00+0800            1.000000                                             app.rr
18   ServiceTest6                   2       2021-03-23 00:00:00+0800   2021-03-23 00:30:00+0800            1.000000                                             app.rr
19   ServiceTest5                   2       2021-03-23 00:00:00+0800   2021-03-23 00:23:00+0800            1.000000                                             app.rr```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Required kernel variables: metric_runs_filtered, metric_runs_summary_full, metric_run_points
if 'metric_runs_filtered' not in globals():
    raise RuntimeError("metric_runs_filtered not found in kernel.")
if 'metric_runs_summary_full' not in globals():
    raise RuntimeError("metric_runs_summary_full not found in kernel.")
if 'metric_run_points' not in globals():
    # create empty placeholder to avoid later errors, but warn by raising
    raise RuntimeError("metric_run_points (per-point run table) not found in kernel.")

# Focus hosts
mysql_hosts = ["Mysql01", "Mysql02"]

# Work on a copy of the filtered-run groups (these already meet max run >=3)
grp = metric_runs_filtered.copy()

# Ensure expected columns exist (fill with NaN if missing)
expect_cols = [
    'cmdb_id','KPI_name','total_anomaly_points','max_consecutive_run_length',
    'first_run_start_timestamp','first_run_end_timestamp','first_run_max_value',
    'global_p95','global_p5','severity_ratio'
]
for c in expect_cols:
    if c not in grp.columns:
        grp[c] = np.nan

# Convert timestamps to timezone-aware datetimes for sorting (they may be strings like "2021-03-23 00:00:00+0800")
grp['_start_dt'] = pd.to_datetime(grp['first_run_start_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)
grp['_end_dt'] = pd.to_datetime(grp['first_run_end_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)

# Compute severity_ratio if absent or NaN: prefer first_run_max_value/global_p95 when possible
def compute_sev_row(r):
    # use existing if numeric
    if pd.notnull(r.get('severity_ratio')):
        try:
            return float(r['severity_ratio'])
        except Exception:
            pass
    maxv = pd.to_numeric(r.get('first_run_max_value', np.nan), errors='coerce')
    gp95 = pd.to_numeric(r.get('global_p95', np.nan), errors='coerce')
    gp5  = pd.to_numeric(r.get('global_p5', np.nan), errors='coerce')
    # If gp95 available and non-zero, use ratio
    if pd.notnull(maxv) and pd.notnull(gp95) and gp95 != 0:
        return float(maxv / gp95)
    # fallback NaN
    return np.nan

grp['severity_ratio_calc'] = grp.apply(compute_sev_row, axis=1)

# Filter to MySQL hosts
mysql_groups_full = grp[grp['cmdb_id'].isin(mysql_hosts)].copy()

# Sort by severity_ratio desc then max_consecutive_run_length desc
mysql_groups_sorted = mysql_groups_full.sort_values(
    by=['severity_ratio_calc','max_consecutive_run_length'],
    ascending=[False, False]
).head(50).reset_index(drop=True)

# Format timestamp strings consistently (Asia/Shanghai)
mysql_groups_sorted['first_run_start_timestamp'] = mysql_groups_sorted['_start_dt'].dt.strftime("%Y-%m-%d %H:%M:%S%z")
mysql_groups_sorted['first_run_end_timestamp'] = mysql_groups_sorted['_end_dt'].dt.strftime("%Y-%m-%d %H:%M:%S%z")

# Round numeric columns for compact display
for c in ['first_run_max_value','global_p95','global_p5','severity_ratio_calc']:
    if c in mysql_groups_sorted.columns:
        mysql_groups_sorted[c] = pd.to_numeric(mysql_groups_sorted[c], errors='coerce').round(6)

# Select desired output columns and rename severity column
mysql_groups_out = mysql_groups_sorted[[
    'cmdb_id','KPI_name','total_anomaly_points','max_consecutive_run_length',
    'first_run_start_timestamp','first_run_end_timestamp','first_run_max_value',
    'global_p95','global_p5','severity_ratio_calc'
]].rename(columns={'severity_ratio_calc':'severity_ratio'})

# Prepare per-node summary for Mysql01 and Mysql02
summ_rows = []
for node in mysql_hosts:
    node_df = mysql_groups_full[mysql_groups_full['cmdb_id']==node].copy()
    faulty_count = int(node_df['KPI_name'].nunique())
    # earliest and latest across groups
    node_df['_s'] = pd.to_datetime(node_df['first_run_start_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)
    node_df['_e'] = pd.to_datetime(node_df['first_run_end_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)
    earliest = node_df['_s'].min()
    latest = node_df['_e'].max()
    # max severity
    node_df['sev_tmp'] = node_df.apply(compute_sev_row, axis=1)
    max_sev = node_df['sev_tmp'].max()
    # top example KPIs by severity then run length
    node_df_sorted = node_df.sort_values(by=['sev_tmp','max_consecutive_run_length','total_anomaly_points'], ascending=[False,False,False])
    example_kpis = node_df_sorted['KPI_name'].dropna().unique()[:5].tolist()
    summ_rows.append({
        'cmdb_id': node,
        'faulty_group_count': faulty_count,
        'earliest_fault_start_timestamp': earliest.strftime("%Y-%m-%d %H:%M:%S%z") if pd.notnull(earliest) else np.nan,
        'latest_fault_end_timestamp': latest.strftime("%Y-%m-%d %H:%M:%S%z") if pd.notnull(latest) else np.nan,
        'max_severity_ratio': round(float(max_sev),6) if pd.notnull(max_sev) else np.nan,
        'example_top_KPIs': example_kpis
    })

mysql_node_summary = pd.DataFrame(summ_rows)

# Keep full filtered per-point run detection results for Mysql hosts in kernel variable
mysql_run_points = metric_run_points[metric_run_points['cmdb_id'].isin(mysql_hosts)].copy()

# Store full filtered group table for later inspection
mysql_groups_full_table = mysql_groups_full.copy()

# Final compact outputs: top-50 group table and 2-row per-node summary
mysql_groups_out, mysql_node_summary

```
Out[6]:
```
Summary focused on MySQL hosts (compact):

Key per-group highlights (top severity examples)
- Mysql01 — Mysql-MySQL_3306_Innodb buffer pool pages dirty
  - total_anomaly_points: 19, max_consecutive_run_length: 5
  - first run: 2021-03-23 00:01:00+0800 → 00:05:00+0800
  - first_run_max_value: 20.0, global_p95: 15.5, severity_ratio ≈ 1.290323

- Mysql02 — Mysql-MySQL_3306_Innodb buffer pool pages dirty
  - total_anomaly_points: 4, max_consecutive_run_length: 3
  - first run: 2021-03-23 00:24:00+0800 → 00:26:00+0800
  - first_run_max_value: 5965.0, global_p95: 5815.0, severity_ratio ≈ 1.025795

- Mysql02 — Mysql-MySQL_3306_ThreadsConnected
  - sustained full-window anomaly: total_anomaly_points = 31, max_consecutive_run_length = 31
  - first run: 2021-03-23 00:00:00+0800 → 00:30:00+0800
  - first_run_max_value: 23.0, global_p95: 22.5, severity_ratio ≈ 1.022222

Additional examples (from top 50):
- Mysql01 has many groups with sustained runs (some 30+ minutes) for Thread/Query/Buffer/InnoDB metrics (example entries for buffer pool, threads cached, queries, log write requests, table cache hits).
- Several Mysql01 groups show very large numeric values but severity ratios around ~1.0–1.29 (indicating they modestly exceed the group p95).

Per-node summary (compact)
- Mysql01:
  - faulty_group_count: 125
  - earliest_fault_start_timestamp: 2021-03-23 00:01:00+0800
  - latest_fault_end_timestamp: 2021-03-23 00:30:00+0800
  - max_severity_ratio: 1.290323
  - example_top_KPIs (up to 5, highest-severity): includes "Mysql-MySQL_3306_Innodb buffer pool pages dirty" (and many InnoDB/buffer/threads/queries KPIs)

- Mysql02:
  - faulty_group_count: 117
  - earliest_fault_start_timestamp: 2021-03-23 00:00:00+0800
  - latest_fault_end_timestamp: 2021-03-23 00:30:00+0800
  - max_severity_ratio: 1.025795
  - example_top_KPIs (up to 5): includes "Mysql-MySQL_3306_Innodb buffer pool pages dirty", "Mysql-MySQL_3306_ThreadsConnected", etc.

Interpretation / next steps
- Both MySQL nodes have large numbers of sustained faulty KPI groups during the incident window; Mysql01 shows the highest single severity (~1.29).
- Prioritize Mysql01 and Mysql02 for deeper investigation (InnoDB buffer pool, thread/connection metrics, query/log write, table cache).
- Use the stored full results for detailed triage:
  - Group-level and run summaries: metric_runs_summary_full, metric_runs_filtered
  - MySQL-focused group table and per-node summaries: mysql_groups_full_table, mysql_groups_out, mysql_node_summary
  - Per-minute run points for MySQL hosts: mysql_run_points

If you want, I can (a) show more rows from the MySQL group table, (b) display raw per-minute points for a specific KPI, or (c) produce a short prioritized action list for Mysql01.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                         KPI_name  total_anomaly_points  max_consecutive_run_length first_run_start_timestamp   first_run_end_timestamp  first_run_max_value    global_p95   global_p5  severity_ratio
0   Mysql01  Mysql-MySQL_3306_Innodb buffer pool pages dirty                    19                           5  2021-03-23 00:01:00+0800  2021-03-23 00:05:00+0800                 20.0  1.550000e+01    0.000000        1.290323
1   Mysql02  Mysql-MySQL_3306_Innodb buffer pool pages dirty                     4                           3  2021-03-23 00:24:00+0800  2021-03-23 00:26:00+0800               5965.0  5.815000e+03  588.000000        1.025795
2   Mysql02                Mysql-MySQL_3306_ThreadsConnected                    31                          31  2021-03-23 00:00:00+0800  2021-03-23 00:30:00+0800                 23.0  2.250000e+01   22.000000        1.022222
3   Mysql01                  Mysql-MySQL_3306_Threads Cached                    30                          30  2021-03-23 00:01:00+0800  2021-03-23 00:30:00+0800                 69.0  6.800000e+01   68.000000        1.014706
4   Mysql02                      Mysql-MySQL_3306_Sort Range                    15                           3  2021-03-23 00:28:00+0800  2021-03-23 00:30:00+0800              39483.0  3.943700e+04    0.000000        1.001166
..      ...                                              ...                   ...                         ...                       ...                       ...                  ...           ...         ...             ...
45  Mysql01               Mysql-MySQL_3306_Qcache Not Cached                     4                           4  2021-03-23 00:26:00+0800  2021-03-23 00:29:00+0800          102252161.0  1.022511e+08    7.460915        1.000010
46  Mysql01                      Mysql-MySQL_3306_Com Select                     4                           4  2021-03-23 00:26:00+0800  2021-03-23 00:29:00+0800          102267306.0  1.022662e+08    7.460915        1.000010
47  Mysql01       Mysql-MySQL_3306_Innodb log write requests                     4                           4  2021-03-23 00:26:00+0800  2021-03-23 00:29:00+0800            4675257.0  4.675208e+06    0.314985        1.000010
48  Mysql01                  Mysql-MySQL_3306_MySQL  Queries                     4                           4  2021-03-23 00:26:00+0800  2021-03-23 00:29:00+0800          224207304.0  2.242050e+08   16.425460        1.000010
49  Mysql01           Mysql-MySQL_3306_Table open cache hits                     4                           4  2021-03-23 00:26:00+0800  2021-03-23 00:29:00+0800          124913817.0  1.249125e+08    9.118670        1.000010

[50 rows x 10 columns],    cmdb_id  faulty_group_count earliest_fault_start_timestamp latest_fault_end_timestamp  max_severity_ratio                                   example_top_KPIs
0  Mysql01                 125       2021-03-23 00:01:00+0800   2021-03-23 00:30:00+0800            1.290323  [Mysql-MySQL_3306_Innodb buffer pool pages dir...
1  Mysql02                 117       2021-03-23 00:00:00+0800   2021-03-23 00:30:00+0800            1.025795  [Mysql-MySQL_3306_Innodb buffer pool pages dir...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```